In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path



DATA_DIR = Path('../data/raw')
DATA_OUT = Path('../data/processed')
DATA_OUT.mkdir(exist_ok=True)


In [2]:
# load files
demo_l = pd.read_sas(DATA_DIR / 'DEMO_L.xpt', format='xport', encoding='utf-8')
ghb_l  = pd.read_sas(DATA_DIR / 'GHB_L.xpt',  format='xport', encoding='utf-8')
diq_l  = pd.read_sas(DATA_DIR / 'DIQ_L.xpt',  format='xport', encoding='utf-8')
bmx_l  = pd.read_sas(DATA_DIR / 'BMX_L.xpt',  format='xport', encoding='utf-8')
demo_p = pd.read_sas(DATA_DIR / 'P_DEMO.xpt', format='xport', encoding='utf-8')
ghb_p  = pd.read_sas(DATA_DIR / 'P_GHB.xpt',  format='xport', encoding='utf-8')
diq_p  = pd.read_sas(DATA_DIR / 'P_DIQ.xpt',  format='xport', encoding='utf-8')
bmx_p  = pd.read_sas(DATA_DIR / 'P_BMX.xpt',  format='xport', encoding='utf-8')

In [3]:
# merge and stack cycles
def merge_cycle(demo, ghb, diq, bmx, cycle_label):
    df = demo.merge(ghb, on='SEQN', how='left')
    df = df.merge(diq, on='SEQN', how='left')
    df = df.merge(bmx, on='SEQN', how='left')
    df['cycle'] = cycle_label
    return df

cycle_l = merge_cycle(demo_l, ghb_l, diq_l, bmx_l, '2021-2023')
cycle_p = merge_cycle(demo_p, ghb_p, diq_p, bmx_p, '2019-2020')

print(f"2021-2023 shape: {cycle_l.shape}")
print(f"2019-2020 shape: {cycle_p.shape}")

2021-2023 shape: (11933, 59)
2019-2020 shape: (15560, 79)


In [4]:
# Find columns common to both cycles
common_cols = list(set(cycle_l.columns) & set(cycle_p.columns))
common_cols.sort()

print(f"Columns in L only: {set(cycle_l.columns) - set(cycle_p.columns)}")
print(f"Columns in P only: {set(cycle_p.columns) - set(cycle_l.columns)}")
print(f"Common columns: {len(common_cols)}")

# Stack using only common columns
df = pd.concat([cycle_l[common_cols], cycle_p[common_cols]], ignore_index=True)
print(f"\nCombined shape: {df.shape}")

Columns in L only: {'WTMEC2YR', 'RIDEXAGM', 'DMDYRUSR', 'DMDHRGND', 'WTINT2YR', 'DMDHREDZ', 'DMDHRAGZ', 'DMDHSEDZ', 'DMQMILIZ', 'DMDHRMAZ', 'DMDHHSIZ', 'WTPH2YR'}
Columns in P only: {'DIQ080', 'SIAINTRP', 'DID260', 'MIAPROXY', 'DIQ360', 'WTMECPRP', 'DMDYRUSZ', 'DIQ260U', 'DIQ275', 'DID250', 'MIAINTRP', 'DID310D', 'WTINTPRP', 'DIQ230', 'DID330', 'MIALANG', 'FIAINTRP', 'AIALANGA', 'DID320', 'DID350', 'SIAPROXY', 'DIQ240', 'DIQ300S', 'DID310S', 'FIAPROXY', 'DIQ350U', 'SIALANG', 'DID341', 'DIQ300D', 'FIALANG', 'DIQ291', 'DIQ280'}
Common columns: 47

Combined shape: (27493, 47)


In [5]:
key_cols = ['SEQN', 'cycle', 'DIQ010', 'RIDRETH3', 'LBXGH', 'BMXBMI', 'RIDAGEYR', 'RIAGENDR']

print(df.shape)

print("\nKey column names present:")
print([c for c in key_cols if c in df.columns])

print("\nDiabetes diagnosis (DIQ010) value counts:")
print(df['DIQ010'].value_counts(dropna=False))

print("\nRace/ethnicity (RIDRETH3) value counts:")
print(df['RIDRETH3'].value_counts(dropna=False))

print("\nHbA1c (LBXGH) summary:")
print(df['LBXGH'].describe())

print("\n BMI (BMXBMI) summary:")
print(df['BMXBMI'].describe())

(27493, 47)

Key column names present:
['SEQN', 'cycle', 'DIQ010', 'RIDRETH3', 'LBXGH', 'BMXBMI', 'RIDAGEYR', 'RIAGENDR']

Diabetes diagnosis (DIQ010) value counts:
DIQ010
2.0    23620
1.0     2526
NaN      767
3.0      568
9.0       12
Name: count, dtype: int64

Race/ethnicity (RIDRETH3) value counts:
RIDRETH3
3.0    11488
4.0     5695
1.0     3107
2.0     2917
6.0     2319
7.0     1967
Name: count, dtype: int64

HbA1c (LBXGH) summary:
count    16452.000000
mean         5.742676
std          1.061938
min          2.800000
25%          5.200000
50%          5.500000
75%          5.900000
max         17.100000
Name: LBXGH, dtype: float64

 BMI (BMXBMI) summary:
count    21608.000000
mean        26.888074
std          8.315204
min         11.100000
25%         20.900000
50%         26.100000
75%         31.500000
max         92.300000
Name: BMXBMI, dtype: float64


In [6]:

# Recode: 1 = diabetes, 0 = no diabetes, drop borderline/refused for now
df['diabetes'] = np.where(df['DIQ010'] == 1, 1,
                 np.where(df['DIQ010'] == 2, 0, np.nan))

print("Diabetes distribution")
print(df['diabetes'].value_counts(dropna=False))
print(f"\nClass balance: {df['diabetes'].mean():.1%} diabetic")


# Check across race/ethnicity subgroups
# RIDRETH3: 1=Mexican American, 2=Other Hispanic, 3=Non-Hispanic White,
#            4=Non-Hispanic Black, 6=Non-Hispanic Asian, 7=Other/Multi
race_labels = {1:'Mexican American', 2:'Other Hispanic', 3:'NH White',
               4:'NH Black', 6:'NH Asian', 7:'Other/Multi'}
df['race_label'] = df['RIDRETH3'].map(race_labels)

print("\nCycle distribution:")
print(df['cycle'].value_counts())

print("\n Diabetes rate by race/ethnicity:")
print(df.groupby('race_label')['diabetes'].agg(['mean','sum','count']).round(3))

Diabetes distribution
diabetes
0.0    23620
1.0     2526
NaN     1347
Name: count, dtype: int64

Class balance: 9.7% diabetic

Cycle distribution:
cycle
2019-2020    15560
2021-2023    11933
Name: count, dtype: int64

 Diabetes rate by race/ethnicity:
                   mean     sum  count
race_label                            
Mexican American  0.094   277.0   2937
NH Asian          0.090   199.0   2206
NH Black          0.112   609.0   5426
NH White          0.093  1018.0  10954
Other Hispanic    0.095   260.0   2749
Other/Multi       0.087   163.0   1874


In [7]:
print("Missing value counts for key columns:")
print(df[key_cols + ['diabetes']].isnull().sum())

os.makedirs('../data/processed', exist_ok=True)

# Save as CSV instead of parquet
df.to_csv('../data/processed/merged_nhanes.csv', index=False)
print("Saved to data/processed/merged_nhanes.csv")


Missing value counts for key columns:
SEQN            0
cycle           0
DIQ010        767
RIDRETH3        0
LBXGH       11041
BMXBMI       5885
RIDAGEYR        0
RIAGENDR        0
diabetes     1347
dtype: int64
Saved to data/processed/merged_nhanes.csv
